# Techniques to Improve Join Performance in Spark

## Real-Time Scenario

Suppose you have:

```
Customer Table

500 Million Records
```

and

```
Country Lookup

250 Records
```

How can you improve join performance?

---

# 1. Use Broadcast Join (Best for Small Tables)

## When to Use

One table is very small (typically less than 10–100 MB, depending on your Spark configuration).

```
Customer Table

500 Million Rows

JOIN

Country Table

250 Rows
```

Instead of shuffling both tables,

broadcast the small table.

```python
from pyspark.sql.functions import broadcast

result = customer_df.join(
    broadcast(country_df),
    "country_code"
)
```

### Benefits

- No shuffle
- Faster execution
- Less network traffic

---

# 2. Filter Data Before Join

❌ Bad

```python
customer.join(order, "customer_id")
.filter(col("status") == "ACTIVE")
```

Spark joins everything first.

---

✅ Better

```python
customer = customer.filter(col("status") == "ACTIVE")

result = customer.join(order, "customer_id")
```

Benefits:

- Smaller dataset
- Less shuffle
- Faster join

---

# 3. Select Only Required Columns

Don't carry unnecessary columns.

❌ Bad

```python
customer.join(order)
```

100 columns move through the network.

---

✅ Better

```python
customer = customer.select(
    "customer_id",
    "name"
)

order = order.select(
    "customer_id",
    "amount"
)
```

Benefits:

- Less memory
- Less serialization
- Faster processing

---

# 4. Partition Data on Join Key

Suppose

```
Join Key

CustomerID
```

Partition by CustomerID.

```python
customer = customer.repartition("customer_id")

order = order.repartition("customer_id")
```

Benefits

- Less shuffle
- Better parallelism

---

# 5. Avoid Data Skew

Example

```
Country

India 95%

USA 3%

UK 2%
```

Most records go to one Executor.

Result

```
Executor 1

10 Hours

Executor 2

5 Minutes

Executor 3

5 Minutes
```

---

Solution

- Salting
- Repartition
- AQE (Adaptive Query Execution)

---

# 6. Use Bucketing (Hive/Spark Tables)

If tables are frequently joined

```
Customer

JOIN

Orders
```

Bucket them.

```sql
CLUSTERED BY (customer_id)
INTO 32 BUCKETS
```

Benefits

- Reduced shuffle
- Faster joins

---

# 7. Enable Adaptive Query Execution (AQE)

Spark can optimize joins automatically.

```python
spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)
```

AQE can:

- Convert sort merge joins into broadcast joins
- Handle skewed joins
- Optimize partition sizes

---

# 8. Cache Reused DataFrames

If a table is used multiple times

```python
customer.cache()
```

Instead of reading from disk repeatedly,

Spark reads from memory.

---

# 9. Increase Parallelism

Too few partitions

```
2 Partitions

Huge Tasks
```

Better

```
200 Partitions

More Parallelism
```

Example

```python
df = df.repartition(200)
```

---

# 10. Use Correct Join Type

Need only matching rows?

Use

```python
inner join
```

instead of

```python
full outer join
```

Smaller result

Faster execution

---

# 11. Remove Duplicate Keys Before Join

Example

```
Country Lookup

US

US

US

IN
```

Duplicates increase join work.

Remove duplicates first.

```python
country = country.dropDuplicates()
```

---

# 12. Optimize File Size

Thousands of small files

```
5 KB

10 KB

20 KB
```

Slow.

Use

```python
repartition()

coalesce()

OPTIMIZE
```

---

# 13. Use Predicate Pushdown

Instead of

```python
customer.join(order)
.filter(col("year")==2025)
```

Use

```python
customer = customer.filter(col("year")==2025)
```

Spark reads fewer records.

---

# 14. Use Delta Optimize (Databricks)

```sql
OPTIMIZE customer
```

Benefits

- Compact files
- Better data skipping
- Faster joins

---

# 15. Use Z-Ordering (Databricks)

```sql
OPTIMIZE customer

ZORDER BY (customer_id)
```

Improves lookup performance on join/filter columns.

---

# 16. Analyze Execution Plan

Always check

```python
df.explain(True)
```

Look for

- BroadcastHashJoin
- SortMergeJoin
- ShuffleExchange
- Cartesian Product

---

# Real-Time Project Example

Suppose

```
Transaction Table

500 Million Records
```

```
Customer Table

20 Million Records
```

```
Country Lookup

250 Records
```

### What We Did

✔ Filtered transactions for the required date.

✔ Selected only needed columns.

✔ Broadcasted the country lookup table.

✔ Repartitioned on `customer_id`.

✔ Enabled AQE.

✔ Cached customer data because it was reused.

✔ Used Delta `OPTIMIZE` and `ZORDER`.

**Result**

- Runtime reduced from **42 minutes to 18 minutes**.
- Shuffle data reduced significantly.
- Cluster CPU utilization improved.

---

# Join Optimization Flow

```text
Read Tables
      │
      ▼
Filter Records Early
      │
      ▼
Select Required Columns
      │
      ▼
Remove Duplicates
      │
      ▼
Broadcast Small Lookup Table
      │
      ▼
Repartition on Join Key
      │
      ▼
Enable AQE
      │
      ▼
Perform Join
      │
      ▼
Cache if Reused
      │
      ▼
Write Output
```

---

# Interview Answer (2–3 Minutes)

> To improve join performance in Spark, I first filter the data and select only the required columns to reduce the amount of data being processed. If one table is small, I use a broadcast join to eliminate shuffle operations. For large tables, I repartition both DataFrames on the join key to improve parallelism and reduce data movement. I also check for data skew and use techniques such as salting or Adaptive Query Execution (AQE) to balance the workload. If the same DataFrame is reused multiple times, I cache it to avoid repeated computation. In Databricks, I use `OPTIMIZE` and `ZORDER` for Delta tables, and I always review the execution plan using `explain()` to verify whether Spark is using an efficient join strategy such as `BroadcastHashJoin` instead of an expensive `SortMergeJoin`.